# Inspecting the Master Table to Assess N-back Feasibility

## What datasets contain trials without removals? 

In [12]:
import pandas as pd
import numpy as np
import os

# Paths
data_path = "/Users/Kurvi001/Documents/Serial_Dependence/Analysis/results/SD_ma_master_table.csv"

# Load preprocessed master table
df = pd.read_csv(data_path, sep=";")

In [ ]:
# Preserve the row order
df["_row"] = np.arange(len(df))
df = df.sort_values("_row").copy()

# Use condition, participant number, and block as grouping variables
group_cols = ["study", "codenum", "obs", "block", "cond"]

# Previous retained trial for the same participant × condition x block x cond
g = df.groupby(group_cols, sort=False)
df["previous_theta"] = g["theta"].shift(1)

# Calculate delta: difference between previous and current theta
df["calculated_delta"] = df["previous_theta"] - df["theta"]

orientation = df["stimulus"] == "Orientation"
motion = df["stimulus"] == "Motion"

# Same logic as sdp_acute_ang
df.loc[orientation & (df["calculated_delta"] > 90), "calculated_delta"] -= 180
df.loc[orientation & (df["calculated_delta"] < -90), "calculated_delta"] += 180

# Equivalent wrapping for motion
df.loc[motion & (df["calculated_delta"] > 180), "calculated_delta"] -= 360
df.loc[motion & (df["calculated_delta"] < -180), "calculated_delta"] += 360

# The master table subsequently changes all theta's (orientation and motion) from -90 to +90
df.loc[(df["calculated_delta"] == -90), "calculated_delta"] = 90

# Only test the match for trials where the previous theta is known (so not the first trials of each block)
df["testable"] = df["previous_theta"].notna() & df["theta"].notna() & df["delta"].notna()
df["match"] = df["testable"] & (df["calculated_delta"] == df["delta"])

summary = (
    df.groupby(group_cols, sort=False)
      .agg(
          total_rows=("delta", "size"),
          comparisons=("testable", "sum"),
          matches=("match", "sum"),
      )
      .reset_index()
)

summary["mismatches"] = summary["comparisons"] - summary["matches"]

# One row per codenum, with its study name
study_summary = (
    summary
    .groupby(["study", "codenum"], sort=False)
    .agg(
        participant_blocks=("mismatches", "size"),
        total_mismatches=("mismatches", "sum"),
        mean_mismatches_per_participant_block=("mismatches", "mean"),
    )
    .reset_index()
)

summary.to_csv("/Users/Kurvi001/Documents/Serial_Dependence/Analysis/results/delta_check_by_participant_condition.csv", index=False)
study_summary.to_csv("/Users/Kurvi001/Documents/Serial_Dependence/Analysis/results/delta_check_by_study_codenum.csv", index=False)

# Keep this too: it shows exactly which trials failed the check.
df.to_csv("/Users/Kurvi001/Documents/Serial_Dependence/Analysis/results/delta_check_trial_details.csv", index=False)


## Resolving mismatches across datasets

In [26]:
# Paths
Houborg_b = ("/Users/Kurvi001/Documents/Serial_Dependence/Analysis/data/14_Houborg_et_al_2023b.csv")
Kondo = ("/Users/Kurvi001/Documents/Serial_Dependence/Analysis/data/15_Kondo_et_al_2022.csv")
Chetverikov = ("/Users/Kurvi001/Documents/Serial_Dependence/Analysis/data/05_Chetverikov_Jehee_2023.csv")
Moon_Kwon = ("/Users/Kurvi001/Documents/Serial_Dependence/Analysis/data/17_Moon_Kwon_2022.csv")
Moon_et_al = ("/Users/Kurvi001/Documents/Serial_Dependence/Analysis/data/18_Moon_et_al_2023.csv")

data_path = ("/Users/Kurvi001/Documents/Serial_Dependence/Analysis/results/SD_ma_master_table.csv")

paths = {
    "houb_b": Houborg_b,
    "kondo": Kondo,
    "chetverikov": Chetverikov,
    "moon_kwon": Moon_Kwon,
    "moon_et_al": Moon_et_al,
}

df = pd.read_csv(data_path, sep=";")

source_dfs = {}
source_master_dfs = {}

for name, path in paths.items():
    source_dfs[name] = pd.read_csv(path, sep=";")

    # Reproduce the master-table preprocessing for missing conditions
    if "cond" not in source_dfs[name].columns:
        source_dfs[name]["cond"] = 1

    # Extract only Chetverikov & Jehee from the master table
    study_names = {
        "houb_b": "Houborg et al. (2023b)",
        "kondo": "Kondo et al. (2022)",
        "chetverikov": "Chetverikov & Jehee (2023)",
        "moon_kwon": "Moon & Kwon (2022)",
        "moon_et_al": "Moon et al. (2023)"
    }

    source_master_dfs[name] = (
        df[df["study"] == study_names[name]]
        .copy()
        .reset_index(drop=True)
    )

    # Reset the source-file index too, so both files start at row 0
    source_dfs[name] = source_dfs[name].reset_index(drop=True)
    # Check that the source and master subsets have the same rows
    # They should contain the same total number of trials.
    assert len(source_dfs[name]) == len(source_master_dfs[name])

    # Condition, block, and stimulus should agree row by row.
    assert np.array_equal(
        source_dfs[name]["cond"].to_numpy(),
        source_master_dfs[name]["cond"].to_numpy(),
    )

    assert np.array_equal(
        source_dfs[name]["block"].to_numpy(),
        source_master_dfs[name]["block"].to_numpy(),
    )

    assert np.array_equal(
        source_dfs[name]["stimulus"].to_numpy(),
        source_master_dfs[name]["stimulus"].to_numpy(),
    )

    # Preserve the row order
    source_dfs[name]["_row"] = np.arange(len(source_dfs[name]))
    source_dfs[name] = (source_dfs[name].sort_values(["expnum", "cond", "obs", "_row"], kind="stable").reset_index(drop=True))

    # Find the previous retained trial
    # Variables that define the participant-level experimental context
    context_cols = ["study", "obs", "cond", "expnum"]

    # Within each context, determine whether the block number has changed
    block_changed = (
        source_dfs[name].groupby(context_cols, sort=False)["block"]
        .transform(lambda block: block.ne(block.shift()))
    )

    # Give every continuous occurrence of a block a new number
    source_dfs[name]["block_unique"] = (
        block_changed.groupby(
            [source_dfs[name][column] for column in context_cols],
            sort=False,
        )
        .cumsum()
    )

    # Use study, original participant number, block, and condition as grouping variables.
    group_cols = ["study", "obs", "cond", "block_unique", "expnum"]
    g = source_dfs[name].groupby(group_cols, sort=False)
    source_dfs[name]["previous_theta"] = g["theta"].shift(1)

    # Calculate delta before rounding
    source_dfs[name]["calculated_delta"] = (source_dfs[name]["previous_theta"] - source_dfs[name]["theta"])

    orientation = source_dfs[name]["stimulus"] == "Orientation"
    motion = source_dfs[name]["stimulus"] == "Motion"

    # Apply the orientation wrapping rule
    source_dfs[name].loc[orientation & (source_dfs[name]["calculated_delta"] > 90), "calculated_delta"] -= 180
    source_dfs[name].loc[orientation & (source_dfs[name]["calculated_delta"] < -90), "calculated_delta"] += 180

    # Apply the equivalent motion wrapping rule
    source_dfs[name].loc[motion & (source_dfs[name]["calculated_delta"] > 180), "calculated_delta"] -= 360
    source_dfs[name].loc[motion & (source_dfs[name]["calculated_delta"] < -180), "calculated_delta"] += 360

    # Round in the same way as MATLAB
    def matlab_round(values):
        # Remove tiny floating-point errors from subtraction
        cleaned_values = values.round(10)

        return np.sign(cleaned_values) * np.floor(np.abs(cleaned_values) + 0.5)

    source_dfs[name]["rounded_calculated_delta"] = matlab_round(source_dfs[name]["calculated_delta"])
    source_dfs[name]["rounded_theta"] = matlab_round(source_dfs[name]["theta"])
    source_dfs[name]["rounded_previous_theta"] = matlab_round(source_dfs[name]["previous_theta"])

    # Match the master table's -90/+90 convention
    source_dfs[name].loc[(source_dfs[name]["rounded_calculated_delta"] == -90), "rounded_calculated_delta"] = 90

    # Add the corresponding master-table information
    source_dfs[name]["codenum"] = source_master_dfs[name]["codenum"]
    source_dfs[name]["obsid"] = source_master_dfs[name]["obsid"]
    source_dfs[name]["master_delta"] = source_master_dfs[name]["delta"]
    source_dfs[name]["master_theta"] = source_master_dfs[name]["theta"]

    # Compare the newly calculated delta with the master delta
    source_dfs[name]["testable"] = (source_dfs[name]["previous_theta"].notna() & source_dfs[name]["theta"].notna() & source_dfs[name]["master_delta"].notna())
    exact_match = (source_dfs[name]["rounded_calculated_delta"] == source_dfs[name]["master_delta"])
    equivalent_180 = (source_dfs[name]["rounded_calculated_delta"].abs().eq(180) & source_dfs[name]["master_delta"].abs().eq(180))
    source_dfs[name]["match"] = (source_dfs[name]["testable"] & (exact_match | equivalent_180))
    
   # Summary per participant × condition × block
    summary_group_cols = ["study", "codenum", "obs", "block", "cond", "expnum", "block_unique"]

    summary = (
        source_dfs[name].groupby(summary_group_cols, sort=False)
        .agg(
            total_rows=("master_delta", "size"),
            comparisons=("testable", "sum"),
            matches=("match", "sum"),
        )
        .reset_index()
    )

    summary["mismatches"] = (summary["comparisons"] - summary["matches"])

    # Summary per codenum
    study_summary = (
        summary.groupby(["study", "codenum"], sort=False)
        .agg(
            participant_blocks=("mismatches", "size"),
            total_comparisons=("comparisons", "sum"),
            total_matches=("matches", "sum"),
            total_mismatches=("mismatches", "sum"),
            mean_mismatches_per_participant_block=(
                "mismatches",
                "mean",
            ),
        )
        .reset_index()
    )

    # Save the new results
    results_path = (f"/Users/Kurvi001/Documents/Serial_Dependence/Analysis/results/study_specific/{name}_check/")
    os.makedirs(results_path, exist_ok=True)

    summary.to_csv(results_path + f"{name}_check_by_participant_block.csv", index=False)
    study_summary.to_csv(results_path + f"{name}_check_by_codenum.csv", index=False)
    source_dfs[name].to_csv(results_path + f"{name}_check_trial_details.csv", index=False)

In [9]:
import pandas as pd
import numpy as np
from paths import results_path


def read_csv_automatically(path):
    return pd.read_csv(
        path,
        sep=None,
        engine="python",
    )


original = read_csv_automatically(
    results_path / "SD_ma_master_table.csv"
)

first_half = read_csv_automatically(
    results_path / "SD_ma_master_table_n_back_first_half.csv"
)

second_half = read_csv_automatically(
    results_path / "SD_ma_master_table_n_back_second_half.csv"
)

halves = pd.concat(
    [first_half, second_half],
    ignore_index=True,
)

group_columns = [
    "study",
    "expnum",
    "cond",
    "obs",
]

for table_name, table in [
    ("original", original),
    ("first_half", first_half),
    ("second_half", second_half),
]:
    missing_columns = [
        column
        for column in group_columns + ["error_ori_deb"]
        if column not in table.columns
    ]

    if missing_columns:
        raise ValueError(
            f"{table_name} is missing these columns: {missing_columns}\n"
            f"Available columns: {table.columns.tolist()}"
        )

original["_trial_number"] = (
    original
    .groupby(group_columns, sort=False)
    .cumcount()
)

halves["_trial_number"] = (
    halves
    .groupby(group_columns, sort=False)
    .cumcount()
)

matching_columns = group_columns + ["_trial_number"]

comparison = halves.merge(
    original[matching_columns + ["error_ori_deb"]],
    on=matching_columns,
    how="outer",
    validate="one_to_one",
    indicator=True,
    suffixes=("_halves", "_original"),
)

print("\nRow matching:")
print(comparison["_merge"].value_counts())

matched = comparison[
    comparison["_merge"] == "both"
].copy()

matched["difference"] = (
    matched["error_ori_deb_halves"]
    - matched["error_ori_deb_original"]
)

same = np.isclose(
    matched["error_ori_deb_halves"],
    matched["error_ori_deb_original"],
    equal_nan=True,
    atol=1e-10,
)

print(f"\nMatched trials: {len(matched)}")
print(f"Equal error_ori_deb values: {same.sum()}")
print(f"Different error_ori_deb values: {(~same).sum()}")
print(f"Maximum absolute difference: {matched['difference'].abs().max()}")

different = matched.loc[
    ~same,
    matching_columns
    + [
        "error_ori_deb_halves",
        "error_ori_deb_original",
        "difference",
    ],
]

print("\nFirst different rows:")
print(different.head(20))

/var/folders/f1/b8ysw_117z536046zqsfn7s80000gn/T/ipykernel_30520/883774284.py:69: UserWarning: You are merging on int and float columns where the float values are not equal to their int representation.
  comparison = halves.merge(



Row matching:
_merge
both          475764
right_only     41832
left_only          0
Name: count, dtype: int64

Matched trials: 475764
Equal error_ori_deb values: 475764
Different error_ori_deb values: 0
Maximum absolute difference: 0.0

First different rows:
Empty DataFrame
Columns: [study, expnum, cond, obs, _trial_number, error_ori_deb_halves, error_ori_deb_original, difference]
Index: []


In [11]:
from paths import results_path
import pandas as pd

df_path = results_path / "fit_p_values_n_back_first_half.csv"
df = pd.read_csv(df_path, sep=";")

X = 6
significant_counts = []

for i in range(1, X + 1):
    count = 0
    for p_value in df[f"delta_{i}"]:
        if p_value < 0.05:
            count += 1

    significant_counts.append(count)

print(significant_counts)

[55, 23, 20, 14, 19, 20]
